# Step 4: Context-Aware Multilingual Emotion Classification

## Goal
Build a context-aware multilingual transformer model (XLM-RoBERTa) that incorporates auxiliary contextual signals (Devotion, Morality, Nature, Symbolism) to improve emotion classification accuracy.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizer, XLMRobertaModel, AdamW, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 4.2 data Loading & Input Representation

In [ ]:
# Paths
DATA_DIR = 'd:/hack4health/Hack4Health/round1'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.xlsx')
VAL_PATH = os.path.join(DATA_DIR, 'val.xlsx')
LABEL_MAP_PATH = os.path.join(DATA_DIR, 'label_map.json')

# Load Data
train_df = pd.read_excel(TRAIN_PATH)
val_df = pd.read_excel(VAL_PATH)

# Load Label Map
with open(LABEL_MAP_PATH, 'r') as f:
    label_map = json.load(f)
    
num_classes = len(label_map)
print(f'Training Samples: {len(train_df)}')
print(f'Validation Samples: {len(val_df)}')
print(f'Number of Classes: {num_classes}')

## 4.4 Context Feature Injection (Context Scoring)
We define 4 small keyword lists for the contexts: Devotion, Morality, Nature, Symbolism.
For each poem, we compute a normalized score (0-1) for each context.

In [ ]:
CONTEXT_KEYWORDS = {
    "devotion": [
        "bhakti", "god", "lord", "worship", "faith", "prayer", "divine", "soul", "sacred", "temple", 
        "krishna", "rama", "shiva", "devi", "jesus", "allah", "grace", "blessing", "chant", "hymn",
        "surrender", "praise", "devotee", "spirit"
    ],
    "morality": [
        "dharma", "duty", "truth", "right", "wrong", "virtue", "justice", "sin", "ethics", "moral",
        "righteous", "honest", "conduct", "law", "principle", "honor", "integrity", "goodness",
        "wisdom", "karma", "noble"
    ],
    "nature": [
        "sun", "moon", "star", "sky", "cloud", "rain", "wind", "river", "ocean", "sea", "mountain",
        "tree", "flower", "forest", "bird", "animal", "season", "spring", "winter", "summer", "autumn",
        "earth", "water", "fire", "air", "lotus", "garden"
    ],
    "symbolism": [
        "dream", "shadow", "mystery", "light", "dark", "illusion", "reflection", "metaphor", "allegory",
        "silence", "whisper", "eternal", "infinite", "void", "mirror", "veil", "path", "journey",
        "vision", "secret", "deep", "hidden"
    ]
}

def compute_context_scores(text):
    text = str(text).lower()
    scores = []
    # Simple keyword matching - can be improved with stemmers if needed, but keeping it lightweight as requested
    words = set(text.split())
    
    for context, keywords in CONTEXT_KEYWORDS.items():
        match_count = sum(1 for k in keywords if k in text) # Checking substring presence usually better for simple matching in Indian langs vs exact token match due to morphology, or just simple word match
        # Normalize: simplistic approach -> min(1.0, count / 3.0) assuming >3 keywords is strong signal
        # Or just raw counts? User said "normalized score (0-1)"
        # Let's use a sigmoid-like or simple max scaling.
        # Given poems can be short, 1 or 2 keywords is significant.
        score = min(1.0, match_count / 2.0) 
        scores.append(score)
    return scores

# Precompute for speed
train_context_scores = np.array([compute_context_scores(t) for t in train_df['cleaned_poem']])
val_context_scores = np.array([compute_context_scores(t) for t in val_df['cleaned_poem']])

print(f"Context Scores Shape: {train_context_scores.shape}")
print(f"Sample Scores: {train_context_scores[0]}")

## 4.2 Dataset Class & Tokenization

In [ ]:
class PoetryDataset(Dataset):
    def __init__(self, texts, context_scores, labels, tokenizer, max_len=128):
        self.texts = texts
        self.context_scores = context_scores
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        ctx_score = self.context_scores[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'context_scores': torch.tensor(ctx_score, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

train_dataset = PoetryDataset(
    train_df['cleaned_poem'].values,
    train_context_scores,
    train_df['label_id'].values,
    tokenizer
)

val_dataset = PoetryDataset(
    val_df['cleaned_poem'].values,
    val_context_scores,
    val_df['label_id'].values,
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

## 4.5 Model Architecture

In [ ]:
class ContextAwareXLMR(nn.Module):
    def __init__(self, n_classes, context_dim=4):
        super(ContextAwareXLMR, self).__init__()
        self.bert = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.drop = nn.Dropout(p=0.3)
        # Feature Fusion: 768 (XLM-R) + 4 (Context Scores)
        self.out = nn.Linear(self.bert.config.hidden_size + context_dim, n_classes)
        
    def forward(self, input_ids, attention_mask, context_scores):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Get CLS token embedding (pooled_output match)
        # XLM-R models don't technically have a pooler in the same way, but we can take the first token
        # outputs.last_hidden_state[:, 0, :] is the [CLS] equivalent (<s>)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        output = self.drop(pooled_output)
        
        # Concatenate Context Scores
        # context_scores shape: [batch, 4]
        combined_features = torch.cat((output, context_scores), dim=1)
        
        return self.out(combined_features)

model = ContextAwareXLMR(n_classes=num_classes)
model = model.to(device)

## 4.5 Training Loop

In [ ]:
EPOCHS = 4 # 3-5 recommended
optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Class Weights for Imbalance
class_counts = train_df['label_id'].value_counts().sort_index().values
total_samples = sum(class_counts)
class_weights = torch.tensor([total_samples / (num_classes * c) for c in class_counts], dtype=torch.float)
class_weights = class_weights.to(device)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0
    
    for d in tqdm(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        context_scores = d["context_scores"].to(device)
        targets = d["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            context_scores=context_scores
        )
        
        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)
        
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            context_scores = d["context_scores"].to(device)
            targets = d["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                context_scores=context_scores
            )
            
            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)
            
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            
    return correct_predictions.double() / n_examples, np.mean(losses)

# Training
history = defaultdict(list)
best_accuracy = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)
    
    train_acc, train_loss = train_epoch(
        model,
        train_loader,
        loss_fn,
        optimizer,
        device,
        scheduler,
        len(train_df)
    )
    
    print(f'Train loss {train_loss} accuracy {train_acc}')
    
    val_acc, val_loss = eval_model(
        model,
        val_loader,
        loss_fn,
        device,
        len(val_df)
    )
    
    print(f'Val   loss {val_loss} accuracy {val_acc}')
    print()
    
    if val_acc > best_accuracy:
        torch.save(model.state_dict(), 'models/xlmr_context_model.bin')
        best_accuracy = val_acc

## 4.6 Evaluation & Reporting

In [ ]:
model.load_state_dict(torch.load('models/xlmr_context_model.bin'))
model = model.to(device)

def get_predictions(model, data_loader):
    model = model.eval()
    predictions = []
    prediction_probs = []
    real_values = []
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            context_scores = d["context_scores"].to(device)
            targets = d["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                context_scores=context_scores
            )
            _, preds = torch.max(outputs, dim=1)
            
            predictions.extend(preds)
            prediction_probs.extend(outputs)
            real_values.extend(targets)
            
    predictions = torch.stack(predictions).cpu()
    prediction_probs = torch.stack(prediction_probs).cpu()
    real_values = torch.stack(real_values).cpu()
    return predictions, prediction_probs, real_values

y_pred, y_pred_probs, y_test = get_predictions(
    model,
    val_loader
)

# Classification Report
print(classification_report(y_test, y_pred, target_names=list(label_map.keys())))

# Save Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(20, 20))
sns.heatmap(cm, annot=False, fmt='d', xticklabels=list(label_map.keys()), yticklabels=list(label_map.keys()))
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.savefig('reports/transformer_confusion_matrix.png')

# Save Metrics
report = classification_report(y_test, y_pred, target_names=list(label_map.keys()), output_dict=True)
with open('reports/transformer_metrics.json', 'w') as f:
    json.dump(report, f, indent=4)